# Train Jeeves's voice: a small style model for Home Mind

This trains **Qwen3-0.6B** to reword Home Mind's plain answers in the persona's voice and emotion, for example *"[disapproving] The furnace filter was changed on March 3, 47 days ago."* → *"The furnace filter was last changed on March 3, 47 days ago. Somewhat overdue, if I may say so."* The result runs in Ollama on your GPU computer as `persona_model`.

1. **Practice set:** made-up plain answers for every emotion, downloaded from Home Mind (System › Persona › *download the practice set*). Nothing personal is in it.
2. **Teacher:** Gemini (your Google AI Pro key) writes the persona's version of each one. Only versions that pass **the same fact check Home Mind uses** are kept: every number and every "not" kept, nothing much longer.
3. **Train** a LoRA on Qwen3-0.6B, score it on held-out answers, and save a GGUF for Ollama.

Runtime › Change runtime type › **T4** (free) is enough; L4 or A100 is faster.

In [ ]:
%pip install -q "torch" "transformers>=4.51" "datasets" "accelerate" "trl>=0.19" "peft>=0.14" "google-genai"

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4, L4 or A100"
print(torch.cuda.get_device_name(0))

## 1. The practice set

In [ ]:
import json
from google.colab import files
up = files.upload()            # jeeves_corpus.json from Home Mind
corpus = json.loads(next(iter(up.values())))
items, PROMPT, STYLE_SYSTEM = corpus["items"], corpus["prompt"], corpus["style_system"]
print(corpus["persona"], len(items), "answers;", sorted({i["emotion"] for i in items}))

## 2. The teacher writes the persona's versions

Add your Gemini API key under the 🔑 **Secrets** panel as `GEMINI_API_KEY`. The fact check below is the same as Home Mind's (`persona.check`): a version that loses a number, loses a "not", drops "Say yes" or runs long is not kept.

In [ ]:
import re, random, time
from collections import Counter
from google import genai
from google.colab import userdata

NUM = re.compile(r"\d+(?:[.:]\d+)?")
NEG = re.compile(r"\b(?:not|no|never|none|nothing|nobody|nowhere|neither|nor|without)\b|n't\b", re.I)

def check(plain, styled, limit=2.0):          # the same rules as Home Mind's persona.check
    a, b = Counter(NUM.findall(plain)), Counter(NUM.findall(styled))
    if any(b[n] < c for n, c in a.items()): return False
    if len(NEG.findall(styled)) < len(NEG.findall(plain)): return False
    if "Say yes" in plain and "Say yes" not in styled: return False
    return len(styled) <= limit * len(plain) + 60

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
TEACHER = "gemini-flash-latest"
shots = "\n".join(f"[{e['emotion']}] {e['plain']}\n=> {e['styled']}" for e in corpus["examples"])
pairs, dropped = [], 0
for i, it in enumerate(items):
    ask = (PROMPT + "\n\n" + STYLE_SYSTEM + "\n\nExamples:\n" + shots +
           f"\n\nNow:\n[{it['emotion']}] {it['plain']}\n=>")
    try:
        out = client.models.generate_content(model=TEACHER, contents=ask).text.strip().splitlines()[0].strip()
    except Exception as e:
        print("skip:", e); time.sleep(2); continue
    out = re.sub(r"^=>\s*", "", out).strip().strip('"')
    if out and out != it["plain"] and check(it["plain"], out):
        pairs.append({"emotion": it["emotion"], "plain": it["plain"], "styled": out})
    else:
        dropped += 1
    if i % 50 == 0: print(i, "kept", len(pairs), "dropped", dropped)
print("kept", len(pairs), "dropped", dropped)
for p in random.sample(pairs, min(8, len(pairs))): print(f"[{p['emotion']}] {p['plain']}\n   -> {p['styled']}")

## 3. Train Qwen3-0.6B (LoRA)

In [ ]:
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

BASE = "Qwen/Qwen3-0.6B"
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16, device_map="auto")

def row(p):                       # the same shape Home Mind sends: system + "[emotion] plain"
    return {"messages": [{"role": "system", "content": PROMPT + "\n\n" + STYLE_SYSTEM},
                         {"role": "user", "content": f"[{p['emotion']}] {p['plain']}"},
                         {"role": "assistant", "content": p["styled"]}]}

random.Random(7).shuffle(pairs)
cut = max(20, len(pairs) // 10)
test, train = pairs[:cut], pairs[cut:]
ds = Dataset.from_list([row(p) for p in train])
args = SFTConfig(output_dir="jeeves-style", num_train_epochs=3, per_device_train_batch_size=8, gradient_accumulation_steps=2,
                 learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.05, logging_steps=20, save_strategy="no",
                 max_length=1024, bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(), report_to=[])
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM")
trainer = SFTTrainer(model=model, args=args, train_dataset=ds, processing_class=tok, peft_config=lora)
trainer.train()

## 4. Score it on answers it hasn't seen

**Passes the check** is what matters: Home Mind uses the plain answer whenever the check fails, so a low number means fewer answers in the persona's voice, never a wrong fact.

In [ ]:
def restyle(emotion, plain):
    msgs = [{"role": "system", "content": PROMPT + "\n\n" + STYLE_SYSTEM}, {"role": "user", "content": f"[{emotion}] {plain}"}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, enable_thinking=False, return_tensors="pt").to(model.device)
    out = model.generate(ids, max_new_tokens=120, do_sample=False)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip().splitlines()[0].strip()

ok = 0
for p in test:
    s = restyle(p["emotion"], p["plain"])
    ok += check(p["plain"], s) and s != p["plain"]
print(f"passes the check: {ok}/{len(test)} = {ok/len(test):.0%}")
for p in test[:8]: print(f"[{p['emotion']}] {p['plain']}\n   -> {restyle(p['emotion'], p['plain'])}")

If it's below about 85%: run the teacher on more answers (download a bigger practice set, e.g. `n=1500`) or train for 4 to 5 epochs.

## 5. Save as GGUF for Ollama

In [ ]:
merged = trainer.model.merge_and_unload()
merged.save_pretrained("jeeves-style-merged")
tok.save_pretrained("jeeves-style-merged")
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
%pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python llama.cpp/convert_hf_to_gguf.py jeeves-style-merged --outtype q8_0 --outfile jeeves-style.gguf
!ls -lh jeeves-style.gguf
files.download("jeeves-style.gguf")

## 6. Run it on the GPU computer

The file is about 650 MB. On the computer that runs Ollama, put it in a folder with a file named `Modelfile` containing:

```
FROM ./jeeves-style.gguf
PARAMETER temperature 0.3
PARAMETER num_predict 120
```

Then run `ollama create jeeves-style -f Modelfile`. In Home Mind's configuration:

```yaml
models:
  - name: jeeves-style
    type: ollama
    url: http://192.168.1.50:11434
    model: jeeves-style
    num_gpu: 0          # the CPU is plenty for a 0.6B model, leaving the GPU to Voicebox
    keep_alive: "-1"
persona_model: jeeves-style
```

Answers it rewords come back with `styled_by: model`. If it's slow (over 1.5 s spoken, 4 s written), away, or changes a fact, the rules style the answer instead.